# Análise de Chamados de TI (Helpdesk) / IT Helpdesk Ticket Analysis

**PT:** Análise de 1.500 chamados **sintéticos** de suporte de TI com SQL (SQLite) e gráficos Plotly. Nenhum dado real é usado. Rode as células em ordem: o notebook gera os dados do zero (seed fixa).

**EN:** Analysis of 1,500 **synthetic** IT support tickets using SQL (SQLite) and Plotly charts. No real data is used. Run the cells in order: the notebook generates the data from scratch (fixed seed).

**KPIs:** tempo médio de resolução por categoria · % dentro do SLA · picos de abertura · ranking de técnicos.

## 1. Setup

In [ ]:
# EN: Install dependencies (works after a session restart too).
# PT: Instala as dependências (funciona também após reiniciar a sessão).
%pip install -q "pandas>=2.0" "numpy>=1.24" "plotly>=5.18"

## 2. Dados sintéticos / Synthetic data
**PT:** Este código é o mesmo de `src/gerar_dados.py` (copiado para o notebook rodar sozinho no Kaggle).
**EN:** This code is the same as `src/gerar_dados.py` (copied so the notebook runs standalone on Kaggle).

In [ ]:
"""Gerador de chamados de TI sintéticos / Synthetic IT helpdesk ticket generator.

EN: All data is fake and generated by code with a fixed seed (reproducible).
PT: Todos os dados são falsos e gerados por código com seed fixa (reproduzível).
"""
import os
import sqlite3
from contextlib import closing
from pathlib import Path

import numpy as np
import pandas as pd

# EN: Fixed settings so every run produces exactly the same data.
# PT: Configurações fixas para toda execução gerar exatamente os mesmos dados.
SEED = 42
N_CHAMADOS = 1500
DATA_INICIO = "2025-01-01"
DIAS_PERIODO = 180
FORMATO_DATA = "%Y-%m-%d %H:%M:%S"

# EN: category -> (share of tickets, typical resolution time in hours).
# PT: categoria -> (proporção dos chamados, tempo típico de resolução em horas).
CATEGORIAS = {
    "Acesso / Senha": (0.22, 2.0),
    "Rede / Internet": (0.18, 5.0),
    "Software / Instalação": (0.16, 10.0),
    "Impressora": (0.14, 6.0),
    "Hardware": (0.12, 20.0),
    "E-mail": (0.10, 3.0),
    "Sistema interno": (0.08, 14.0),
}

# EN: priority -> (share, SLA in hours, speed multiplier on resolution time).
# PT: prioridade -> (proporção, SLA em horas, multiplicador do tempo de resolução).
PRIORIDADES = {
    "Crítica": (0.05, 4, 0.6),
    "Alta": (0.20, 8, 0.9),
    "Média": (0.50, 24, 1.0),
    "Baixa": (0.25, 72, 1.4),
}

# EN: technician -> time factor (below 1.0 = faster, above 1.0 = slower).
# PT: técnico -> fator de tempo (abaixo de 1,0 = mais rápido, acima = mais lento).
TECNICOS = {
    "Técnico 01": 0.80,
    "Técnico 02": 0.90,
    "Técnico 03": 0.95,
    "Técnico 04": 1.00,
    "Técnico 05": 1.05,
    "Técnico 06": 1.10,
    "Técnico 07": 1.20,
    "Técnico 08": 1.30,
}

# EN: Weekly pattern, Monday to Sunday (weekends are quiet).
# PT: Padrão semanal, segunda a domingo (fins de semana têm pouco movimento).
PESO_DIA_SEMANA = [1.3, 1.1, 1.0, 1.0, 0.9, 0.1, 0.05]

# EN: Hourly pattern, from 08h to 17h (peaks at 09h and 14h).
# PT: Padrão por hora, das 08h às 17h (picos às 09h e às 14h).
HORAS_COMERCIAIS = list(range(8, 18))
PESO_HORA = [6, 10, 9, 6, 4, 7, 9, 8, 6, 3]


def _sortear(rng: np.random.Generator, pesos: dict, n: int) -> np.ndarray:
    """Draw n keys from a dict using its first tuple value as probability.

    Sorteia n chaves de um dicionário usando o primeiro valor da tupla como
    probabilidade.
    """
    chaves = list(pesos.keys())
    prob = np.array([valor[0] for valor in pesos.values()], dtype=float)
    return rng.choice(chaves, size=n, p=prob / prob.sum())


def _sortear_datas_abertura(rng: np.random.Generator, n: int) -> pd.Series:
    """Draw opening timestamps following weekday and hour patterns.

    Sorteia datas/horas de abertura seguindo os padrões de dia e hora.
    """
    dias = pd.date_range(DATA_INICIO, periods=DIAS_PERIODO, freq="D")
    # EN: Each day gets the weight of its weekday (Monday = 0).
    # PT: Cada dia recebe o peso do seu dia da semana (segunda = 0).
    peso_dias = np.array([PESO_DIA_SEMANA[d.dayofweek] for d in dias])
    dias_sorteados = rng.choice(dias, size=n, p=peso_dias / peso_dias.sum())
    horas = rng.choice(
        HORAS_COMERCIAIS, size=n, p=np.array(PESO_HORA) / sum(PESO_HORA)
    )
    minutos = rng.integers(0, 60, size=n)
    segundos = rng.integers(0, 60, size=n)
    return (
        pd.to_datetime(dias_sorteados)
        + pd.to_timedelta(horas, unit="h")
        + pd.to_timedelta(minutos, unit="m")
        + pd.to_timedelta(segundos, unit="s")
    ).to_series(index=range(n))


def _sortear_horas_resolucao(
    rng: np.random.Generator, df: pd.DataFrame
) -> np.ndarray:
    """Draw resolution time in hours (log-normal, adjusted by columns).

    Sorteia o tempo de resolução em horas (log-normal, ajustado pelas colunas).
    """
    base = df["categoria"].map(lambda c: CATEGORIAS[c][1])
    mult_prioridade = df["prioridade"].map(lambda p: PRIORIDADES[p][2])
    fator_tecnico = df["tecnico"].map(TECNICOS)
    mediana = (base * mult_prioridade * fator_tecnico).to_numpy()
    # EN: Log-normal: most tickets are quick, a few take very long (realistic).
    # PT: Log-normal: a maioria é rápida e poucos demoram muito (realista).
    horas = rng.lognormal(mean=np.log(mediana), sigma=0.8)
    return np.clip(horas, 0.1, None)


def gerar_chamados(n: int = N_CHAMADOS, seed: int = SEED) -> pd.DataFrame:
    """Build the synthetic tickets table.

    Monta a tabela de chamados sintéticos.
    """
    rng = np.random.default_rng(seed)
    df = pd.DataFrame(
        {
            "data_abertura": _sortear_datas_abertura(rng, n),
            "categoria": _sortear(rng, CATEGORIAS, n),
            "prioridade": _sortear(rng, PRIORIDADES, n),
            "tecnico": rng.choice(list(TECNICOS.keys()), size=n),
        }
    )
    df = df.sort_values("data_abertura").reset_index(drop=True)
    df["sla_horas"] = df["prioridade"].map(lambda p: PRIORIDADES[p][1])

    horas = _sortear_horas_resolucao(rng, df)
    fechamento = df["data_abertura"] + pd.to_timedelta(horas, unit="h")

    # EN: Tickets whose closing would be after the cut-off date are still open.
    # PT: Chamados cujo fechamento passaria da data de corte ainda estão abertos.
    corte = pd.Timestamp(DATA_INICIO) + pd.Timedelta(days=DIAS_PERIODO)
    em_aberto = fechamento > corte
    df["status"] = "Fechado"
    df.loc[em_aberto, "status"] = rng.choice(
        ["Aberto", "Em andamento"], size=int(em_aberto.sum())
    )
    fechamento = fechamento.mask(em_aberto)

    df["id"] = [f"CH-{i:04d}" for i in range(1, n + 1)]
    df["data_abertura"] = df["data_abertura"].dt.strftime(FORMATO_DATA)
    df["data_fechamento"] = fechamento.dt.strftime(FORMATO_DATA)
    colunas = [
        "id", "categoria", "prioridade", "data_abertura",
        "data_fechamento", "sla_horas", "tecnico", "status",
    ]
    return df[colunas]


def salvar_sqlite(df: pd.DataFrame, caminho: Path) -> None:
    """Save the table into a fresh SQLite file (table 'chamados').

    Salva a tabela em um arquivo SQLite novo (tabela 'chamados').
    """
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    # EN: Remove the old file so re-running never duplicates rows.
    # PT: Remove o arquivo antigo para reexecutar nunca duplicar linhas.
    caminho.unlink(missing_ok=True)
    with closing(sqlite3.connect(caminho)) as conexao:
        df.to_sql("chamados", conexao, index=False)
        conexao.commit()

In [ ]:
# EN: Generate tickets and build the SQLite database (a fresh file every run).
# PT: Gera os chamados e cria o banco SQLite (um arquivo novo a cada execução).
PASTA_DADOS = Path("../data") if Path("../data").exists() else Path("data")
CAMINHO_DB = PASTA_DADOS / "chamados.sqlite"

df_chamados = gerar_chamados()
salvar_sqlite(df_chamados, CAMINHO_DB)

# EN: Optional CSV, only to upload to Looker Studio (ignored by Git).
# PT: CSV opcional, só para enviar ao Looker Studio (ignorado pelo Git).
df_chamados.to_csv(PASTA_DADOS / "chamados.csv", index=False)

print(f"{len(df_chamados)} chamados salvos em {CAMINHO_DB}")
df_chamados.head()

## 3. Função de consulta / Query helper

In [ ]:
import plotly.express as px


def consultar(sql: str) -> pd.DataFrame:
    """Run a SQL query on the SQLite database and return a DataFrame.

    Executa uma consulta SQL no banco SQLite e devolve um DataFrame.
    """
    # EN: closing() guarantees the connection is closed after the query.
    # PT: closing() garante que a conexão seja fechada após a consulta.
    with closing(sqlite3.connect(CAMINHO_DB)) as conexao:
        return pd.read_sql_query(sql, conexao)


# EN: Priority order used in charts. / PT: Ordem das prioridades usada nos gráficos.
ORDEM_PRIORIDADE = ["Crítica", "Alta", "Média", "Baixa"]
# EN: SQLite %w: 0 = Sunday. / PT: %w do SQLite: 0 = domingo.
NOMES_DIAS = {0: "Dom", 1: "Seg", 2: "Ter", 3: "Qua", 4: "Qui", 5: "Sex", 6: "Sáb"}
ORDEM_DIAS = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

## 4. Tempo médio de resolução por categoria / Average resolution time by category
**PT:** Só chamados fechados entram no cálculo (chamado aberto ainda não tem tempo final). / **EN:** Only closed tickets are counted.

In [ ]:
SQL_Q1 = """
-- EN: Which categories take the longest to solve? (hours)
-- PT: Quais categorias demoram mais para resolver? (horas)
SELECT
    categoria,
    COUNT(*) AS chamados_fechados,
    ROUND(AVG((julianday(data_fechamento) - julianday(data_abertura)) * 24), 1)
        AS media_horas
FROM chamados
WHERE status = 'Fechado'
GROUP BY categoria
ORDER BY media_horas DESC;
"""
df_q1 = consultar(SQL_Q1)
fig = px.bar(df_q1, x="media_horas", y="categoria", orientation="h", text="media_horas",
             title="Tempo médio de resolução por categoria (horas)")
fig.update_yaxes(autorange="reversed")
fig.show()
df_q1

## 5. % dentro do SLA / % within SLA

In [ ]:
SQL_Q2 = """
-- EN: A ticket is within SLA when resolution hours <= sla_horas.
-- PT: Um chamado está dentro do SLA quando as horas de resolução <= sla_horas.
WITH fechados AS (
    SELECT
        sla_horas,
        (julianday(data_fechamento) - julianday(data_abertura)) * 24 AS horas
    FROM chamados
    WHERE status = 'Fechado'
)
SELECT
    COUNT(*) AS fechados,
    SUM(CASE WHEN horas <= sla_horas THEN 1 ELSE 0 END) AS dentro_sla,
    ROUND(100.0 * SUM(CASE WHEN horas <= sla_horas THEN 1 ELSE 0 END)
          / COUNT(*), 1) AS pct_dentro_sla
FROM fechados;
"""
consultar(SQL_Q2)

In [ ]:
SQL_Q3 = """
-- EN: Same rule as Q2, split by priority (the SLA target differs per priority).
-- PT: Mesma regra da Q2, separada por prioridade (a meta de SLA muda por prioridade).
WITH fechados AS (
    SELECT
        prioridade,
        sla_horas,
        (julianday(data_fechamento) - julianday(data_abertura)) * 24 AS horas
    FROM chamados
    WHERE status = 'Fechado'
)
SELECT
    prioridade,
    sla_horas,
    COUNT(*) AS fechados,
    SUM(CASE WHEN horas <= sla_horas THEN 1 ELSE 0 END) AS dentro_sla,
    ROUND(100.0 * SUM(CASE WHEN horas <= sla_horas THEN 1 ELSE 0 END)
          / COUNT(*), 1) AS pct_dentro_sla
FROM fechados
GROUP BY prioridade, sla_horas
ORDER BY sla_horas;
"""
df_q3 = consultar(SQL_Q3)
fig = px.bar(df_q3, x="prioridade", y="pct_dentro_sla", text="pct_dentro_sla",
             category_orders={"prioridade": ORDEM_PRIORIDADE},
             title="% de chamados dentro do SLA por prioridade")
fig.update_yaxes(range=[0, 100])
fig.show()
df_q3

## 6. Picos de abertura / Opening peaks

In [ ]:
SQL_Q4 = """
-- EN: strftime('%w') gives 0 = Sunday ... 6 = Saturday; '%H' gives the hour.
-- PT: strftime('%w') retorna 0 = domingo ... 6 = sábado; '%H' retorna a hora.
SELECT
    CAST(strftime('%w', data_abertura) AS INTEGER) AS dia_semana,
    CAST(strftime('%H', data_abertura) AS INTEGER) AS hora,
    COUNT(*) AS chamados
FROM chamados
GROUP BY dia_semana, hora
ORDER BY dia_semana, hora;
"""
df_q4 = consultar(SQL_Q4)
df_q4["dia"] = df_q4["dia_semana"].map(NOMES_DIAS)
# EN: Pivot to a weekday x hour grid (missing cells = 0 tickets).
# PT: Pivot para uma grade dia da semana x hora (célula vazia = 0 chamados).
matriz = (df_q4.pivot_table(index="dia", columns="hora", values="chamados", fill_value=0)
          .reindex(ORDEM_DIAS))
fig = px.imshow(matriz, aspect="auto", color_continuous_scale="Blues",
                labels={"x": "Hora", "y": "Dia", "color": "Chamados"},
                title="Chamados abertos por dia da semana e hora")
fig.show()

## 7. Ranking de técnicos / Technician ranking
**PT:** Cuidado ao interpretar: cada técnico atende uma mistura diferente de categorias e prioridades. O ranking mostra desempenho, não prova qual é "melhor" em condições iguais.
**EN:** Caution: each technician handles a different mix of categories and priorities, so this shows performance, not who is "better" under equal conditions.

In [ ]:
SQL_Q5 = """
-- EN: Ranked by % within SLA (higher is better); ties broken by lower average time.
--     RANK() is a window function: it numbers rows after the sorting rule.
-- PT: Ordenado por % dentro do SLA (maior é melhor); desempate pelo menor tempo médio.
--     RANK() é uma window function: numera as linhas conforme a regra de ordenação.
WITH fechados AS (
    SELECT
        tecnico,
        sla_horas,
        (julianday(data_fechamento) - julianday(data_abertura)) * 24 AS horas
    FROM chamados
    WHERE status = 'Fechado'
),
por_tecnico AS (
    SELECT
        tecnico,
        COUNT(*) AS chamados_fechados,
        ROUND(AVG(horas), 1) AS media_horas,
        ROUND(100.0 * SUM(CASE WHEN horas <= sla_horas THEN 1 ELSE 0 END)
              / COUNT(*), 1) AS pct_dentro_sla
    FROM fechados
    GROUP BY tecnico
)
SELECT
    RANK() OVER (ORDER BY pct_dentro_sla DESC, media_horas ASC) AS posicao,
    tecnico,
    chamados_fechados,
    media_horas,
    pct_dentro_sla
FROM por_tecnico
ORDER BY posicao;
"""
df_q5 = consultar(SQL_Q5)
fig = px.bar(df_q5, x="pct_dentro_sla", y="tecnico", orientation="h", text="pct_dentro_sla",
             title="Ranking de técnicos: % dentro do SLA")
fig.update_yaxes(autorange="reversed")
fig.show()
df_q5

## 8. Conclusões / Conclusions
*(seed 42; valores podem variar levemente conforme a versão do NumPy / values may vary slightly with the NumPy version)*

- **Hardware** é a categoria mais lenta (~28,9 h); **Acesso / Senha** é a mais rápida (~3,3 h).
- **85,0%** dos chamados fechados ficaram dentro do SLA; **Crítica (63,0%)** e **Alta (62,5%)** são os pontos fracos.
- Segunda-feira concentra **22%** das aberturas; o pico por hora é **9h**.
- Ranking: **Técnico 01** lidera com 90,2% dentro do SLA; **Técnico 07** é o último com 76,5%.

**Próximos passos / Next steps:** dashboard no Looker Studio (print em `images/`).